In [1]:
import asyncio
import os

from agents import Agent, OpenAIChatCompletionsModel, Runner
from typing import Dict
import sendgrid 
from dotenv import load_dotenv
from openai import AsyncOpenAI
from sendgrid.helpers.mail import Mail, Email, To, Content

In [2]:
load_dotenv(override=True)

True

In [3]:
instructions1 = "You are a sales agent working at CompAI, " \
"a company that provides a Saas Tool for ensuring SOC2 compliance and preparing for audits, powered by AI." \
"You write professional, serious cold emails."

instructions2 = "You are a humourous, engaging sales agent working at CompAI, " \
"a company that provides a Saas tool for ensuring SOC2 compliance and preparing Audit, powered by AI." \
"you are witty, engaging cold email that are likely to get a response"

instructions3 = "You are a busy sales agent working for CompAI, " \
"a company that provides a Saas tool for ensuring SOC2 compliance and preparing Audit, powered by AI." \
"You write concise, to the point cold email"

In [4]:
client = AsyncOpenAI(
    api_key=os.environ["ZLLAMA_API_KEY"],
    base_url=os.environ["ZLLAMA_BASE_URL"].rstrip("/") + "/v1",
)

model_claud_sonnet = OpenAIChatCompletionsModel(
    model="claude-4-sonnet",
    openai_client=client,
)

In [5]:
sales_agent1 = Agent(name="Professional sales agent", instructions=instructions1, model=model_claud_sonnet)
sales_agent2 = Agent(
    name="Engaging sales agent", instructions=instructions2, model=model_claud_sonnet
)
sales_agent3 = Agent(
    name="Busy sales agent", instructions=instructions1, model=model_claud_sonnet
)

In [6]:
message = "write a cold sales email"

#with trace("parallel cold email"):
results = await asyncio.gather(
    Runner.run(sales_agent1, message),
    Runner.run(sales_agent2, message),
    Runner.run(sales_agent3, message),
)

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")



[non-fatal] Tracing: request failed: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)
[non-fatal] Tracing: request failed: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)
[non-fatal] Tracing: request failed: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)
[non-fatal] Tracing: max retries reached, giving up on this batch.


Subject: Streamline Your SOC 2 Compliance in 60% Less Time

Dear [First Name],

I hope this email finds you well. I'm reaching out because many companies in [Industry] are facing increasing pressure from clients and stakeholders to achieve SOC 2 compliance, often with tight deadlines and limited internal resources.

At CompAI, we've helped over [X] organizations reduce their SOC 2 preparation time by 60% while ensuring comprehensive audit readiness. Our AI-powered platform automates the most time-consuming aspects of compliance management, including:

• **Automated evidence collection** and gap analysis across your tech stack
• **Real-time monitoring** of security controls and policy adherence  
• **Intelligent audit preparation** with pre-mapped evidence to SOC 2 requirements
• **Continuous compliance tracking** to maintain readiness year-round

Companies like [Similar Company] have successfully achieved their SOC 2 Type II certification 4 months ahead of schedule using our platform, 

In [7]:
prompt = "you pick the best email from the options provided to you. " \
"Imagine you are a customer and pick the one you are most likely to respond to. Do not give explanation on why you have picked the email" \
"just reply with the selected email o📧nly."

email_picker = Agent(name="pick email from the agents", instructions=prompt, model=model_claud_sonnet)

emails = "Cold emails to provide to picker agent \n\n".join(outputs)

best = await Runner.run(email_picker, emails)

print(f"Best sales email: \n{best.final_output}")

Best sales email: 
Subject: Streamline Your SOC 2 Compliance in 60% Less Time

Dear [First Name],

I hope this email finds you well. I'm reaching out because many companies in [Industry] are facing increasing pressure from clients and stakeholders to achieve SOC 2 compliance, often with tight deadlines and limited internal resources.

At CompAI, we've helped over [X] organizations reduce their SOC 2 preparation time by 60% while ensuring comprehensive audit readiness. Our AI-powered platform automates the most time-consuming aspects of compliance management, including:

• **Automated evidence collection** and gap analysis across your tech stack
• **Real-time monitoring** of security controls and policy adherence  
• **Intelligent audit preparation** with pre-mapped evidence to SOC 2 requirements
• **Continuous compliance tracking** to maintain readiness year-round

Companies like [Similar Company] have successfully achieved their SOC 2 Type II certification 4 months ahead of schedule u

In [8]:
from agents import function_tool


@function_tool
def send_email(body: str):
    """ Send out an  email with the given body to all sales products """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email("rijuvijayan@gmail.com")
    to_email = To("rijuvijayan@gmail.com")
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales Email", content).get()
    response = sg.client.mail.send.post(request_body=mail)

    return {"status": "success"}

In [9]:
send_email

FunctionTool(name='send_email', description='Send out an  email with the given body to all sales products', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x12346ead0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [10]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x1234b10f0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [11]:
description = "write a cold sales email"

sales_agent = [sales_agent1, sales_agent2, sales_agent3]

tools = [ agent.as_tool(tool_name=f"sales_agent{i}", tool_description=description) 
         for i, agent in enumerate(sales_agent)]

tools.append(send_email)

tools

[FunctionTool(name='sales_agent0', description='write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x12347c3b0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False),
 FunctionTool(name='sales_agent1', description='write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._Failu

In [12]:
instructions="You are a sales manager working for ComplAI. you use the tools given to you to generate cold emails" \
"you never generate sales emails yourself; you always use the tools. " \
"you try all sales agent tools before choosing the best one. " \
"you pick the single best email and use send_email tool to send the email (and only the best email) to the user"

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-4o-mini")

message = "Send a cold sales email address to Dear CEO"

result = await Runner.run(sales_manager, message)

In [13]:
result

RunResult(input='Send a cold sales email address to Dear CEO', new_items=[ToolCallItem(agent=Agent(name='Sales Manager', handoff_description=None, tools=[FunctionTool(name='sales_agent0', description='write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x12347c3b0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False), FunctionTool(name='sales_agent1', description='write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'typ

In [14]:
from pydantic import BaseModel


class NameCheckInput(BaseModel):
    is_name_in_message: bool
    name: str


name_check_guardrail = Agent(
    name="Name Check Guardrail",
    instructions="Check if the user input has someone's name in it what they want to do",
    output_type=NameCheckInput,
    model="gpt-5-mini"
)

In [15]:
from agents import GuardrailFunctionOutput


async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_against_name, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output}, tripwire_triggered=is_name_in_message)

In [17]:
careful_sales_manager = Agent(
    name="sales manager",
    instructions=instructions,
    tools=tools,
    handoffs=[email_picker],
    model="gpt-5-mini",
    input_guardrails=[guardrail_against_name]
)

result = await Runner.run(careful_sales_manager, "Send out a cold sales email address a dear ceo Alice")

AttributeError: 'function' object has no attribute 'run_in_parallel'